# 03 · Data quality assessment

## Purpose

Measure missingness, duplicate rows and identifiers, numeric outlier flags, schema roles, and class balance from the actual active CSV.

## Method

Create a quality report. Outliers use the 1.5×IQR screening rule only; data is not deleted automatically. Record target imbalance only when the target exists.

## Interpretation

Use the displayed tables, figures, and saved metrics produced by this run. Do not fill this section with assumed outcomes. For every result, distinguish observed evidence from interpretation.

## Limitations

IQR flags are review indicators, not proof that a value is invalid. Leakage and feature reliability require documentation beyond automated name and type checks.

## References

Quality choices and leakage limitations are documented in docs/implementation_plan.md.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'returnwise').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display
from returnwise.research_state import load_state, save_state
state = load_state()
print('Project:', ROOT)
print('Notebook:', '03 · Data quality assessment')


Project: C:\Users\Lenovo\Documents\Who is back
Notebook: 03 · Data quality assessment


In [2]:
from returnwise.data.profile import quality_report
from returnwise.storage import load_active_dataset
import matplotlib.pyplot as plt

frame, manifest = load_active_dataset()
if frame is None:
    raise RuntimeError('No active dataset. Complete notebook 01 first.')
quality = quality_report(frame)
print({key: value for key, value in quality.items() if key not in {'missing', 'schema', 'target_counts'}})
display(quality['missing'])
display(quality['schema'])
if quality['target_counts'] is not None:
    display(quality['target_counts'])
missing = quality['missing'].head(20)
if not missing.empty:
    ax = missing.sort_values('missing_percent').plot.barh(x='column', y='missing_percent', figsize=(8, max(3, len(missing) * 0.25)), legend=False, color='#B56B33', title='Missingness by feature')
    ax.set_xlabel('Missing values (%)')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values were observed in the active file.')


{'rows': 200000, 'columns': 16, 'duplicate_rows': 0, 'duplicate_rows_percent': 0.0, 'identifier_columns': ['order_id'], 'duplicate_identifiers': {'order_id': 0}, 'iqr_outlier_counts': {'customer_age': 0, 'product_price': 11689, 'discount_percent': 0, 'product_rating': 0, 'past_purchase_count': 23432, 'past_return_rate': 7260, 'delivery_delay_days': 9053, 'session_length_minutes': 0, 'num_product_views': 0, 'used_coupon': 0, 'returned': 0, 'order_id': 0}, 'invalid_values': {'product_price': {'issue': 'Negative value where the documented quantity is non-negative', 'rows': 27285}, 'session_length_minutes': {'issue': 'Negative value where the documented quantity is non-negative', 'rows': 1487}, 'num_product_views': {'issue': 'Negative value where the documented quantity is non-negative', 'rows': 21443}, 'discount_percent': {'issue': 'Discount percent outside 0–100', 'rows': 1686}, 'product_rating': {'issue': 'Rating outside the common 0–5 screening range; confirm source scale', 'rows': 383

,column,missing_rows,missing_percent


,column,role,dtype,non_null,missing,missing_percent,unique_values,examples_from_file,description
0,customer_age,numeric feature,int64,200000,0,0.0,78,"25, 76, 40, 62",Customer age (competition description).
1,product_price,numeric feature,float64,200000,0,0.0,199800,"11.66613697698637, 31.37896692422914, 25.46060...",Product price at purchase (competition descrip...
2,discount_percent,numeric feature,float64,200000,0,0.0,199907,"43.55960142346256, 66.52603939172272, 20.76237...",Discount applied to the product (competition d...
3,product_rating,numeric feature,float64,200000,0,0.0,199936,"1.9335331058958345, 0.6902369777604651, 2.3476...",Average product rating (competition description).
4,past_purchase_count,numeric feature,int64,200000,0,0.0,26,"8, 9, 14, 10",Number of the customer's past purchases (compe...
5,past_return_rate,numeric feature,float64,200000,0,0.0,199934,"0.3314077354329958, 0.10520314818028009, 0.236...",Historical customer return rate (competition d...
6,delivery_delay_days,numeric feature,float64,200000,0,0.0,199947,"-0.08963734587823691, 3.63927684386212, 0.2940...",Delay against estimated delivery (competition ...
7,session_length_minutes,numeric feature,float64,200000,0,0.0,199921,"3.273029019390499, 130.51312301464336, 92.7809...",Duration of the browsing session in minutes (c...
8,num_product_views,numeric feature,int64,200000,0,0.0,50,"26, 8, -2, 7",Products viewed during the session (competitio...
9,device_type,categorical feature,str,200000,0,0.0,3,"tablet, mobile, desktop",Device category used for the session (competit...


,class,rows,percent
0,0,105081,52.5405
1,1,94919,47.4595


No missing values were observed in the active file.
